In [ ]:
#!pip3 install twilio

In [1]:
import os
from twilio.rest import Client
from twilio_config import *
import time

from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json


import pandas as pd
import requests
from bs4  import BeautifulSoup
from tqdm import tqdm

from datetime import datetime


# Armado de la URL

In [2]:
query = 'Tarapoto'
api_key = API_KEY_WAPI

url_clima = 'http://api.weatherapi.com/v1/forecast.json?key='+api_key+'&q='+query+'&days=1&aqi=no&alerts=no'
url_clima

'http://api.weatherapi.com/v1/forecast.json?key=28270ea19dc5497ba8004605260807&q=Tarapoto&days=1&aqi=no&alerts=no'

In [3]:
response = requests.get(url_clima).json()

In [ ]:
response

In [ ]:
response.keys()

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]['time'].split()[0]

In [ ]:
response['forecast']['forecastday'][0].keys()

In [ ]:
len(response['forecast']['forecastday'][0]['hour'])

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]['time'].split()[0]

In [ ]:
response['forecast']['forecastday'][0]['hour'][1]['time'].split()[1].split(':')[0]

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]['condition']['text']

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]['will_it_rain']

In [ ]:
response['forecast']['forecastday'][0]['hour'][0]['chance_of_rain']

# Dataframe

In [4]:
def get_forecast(response,i):
    
    fecha = response['forecast']['forecastday'][0]['hour'][i]['time'].split()[0]
    hora = int(response['forecast']['forecastday'][0]['hour'][i]['time'].split()[1].split(':')[0])
    condicion = response['forecast']['forecastday'][0]['hour'][i]['condition']['text']
    tempe = float(response['forecast']['forecastday'][0]['hour'][i]['temp_c'])
    rain = response['forecast']['forecastday'][0]['hour'][i]['will_it_rain']
    prob_rain = response['forecast']['forecastday'][0]['hour'][i]['chance_of_rain']
    
    return fecha,hora,condicion,tempe,rain,prob_rain

In [5]:
datos = []

for i in tqdm(range(len(response['forecast']['forecastday'][0]['hour'])),colour = 'green'):
    
    datos.append(get_forecast(response,i))
    

100%|██████████| 24/24 [00:00<00:00, 363405.40it/s]


In [ ]:
datos

In [6]:
col = ['Fecha','Hora','Condicion','Temperatura','Lluvia','prob_lluvia']
df = pd.DataFrame(datos,columns=col)
df = df.sort_values(by = 'Hora',ascending = True)
df

,Fecha,Hora,Condicion,Temperatura,Lluvia,prob_lluvia
0,2026-07-07,0,Light rain shower,22.1,0,47
1,2026-07-07,1,Patchy rain nearby,22.1,0,54
2,2026-07-07,2,Patchy rain nearby,21.9,0,66
3,2026-07-07,3,Light rain shower,21.8,0,70
4,2026-07-07,4,Patchy rain nearby,21.4,0,64
5,2026-07-07,5,Patchy light rain,21.4,1,73
6,2026-07-07,6,Light rain shower,21.3,1,71
7,2026-07-07,7,Patchy rain nearby,21.7,0,59
8,2026-07-07,8,Light rain shower,22.7,0,36
9,2026-07-07,9,Patchy light rain,24.3,1,71


In [ ]:
df[df['Lluvia']==1]

In [7]:
df_rain =  df[(df['Lluvia']==1) & (df['Hora']>6) & (df['Hora']< 22)]
df_rain = df_rain[['Hora','Condicion']]
df_rain.set_index('Hora', inplace = True)

In [ ]:
df['Fecha'][0]

In [ ]:
df_rain

In [ ]:
f"""🌦️ Hola!

Aquí tienes el pronóstico de lluvias para hoy, {df['Fecha'][0]}, en {query}:

⏰ Horas con lluvia pronosticada:
{df_rain.to_string() if not df_rain.empty else 'No se esperan lluvias importantes durante el día.'}

¡Que tengas un excelente día! ☔️😊
"""

'🌦️ Hola!\n\nAquí tienes el pronóstico de lluvias para hoy, 2026-07-07, en Tarapoto:\n\n⏰ Horas con lluvia pronosticada:\n              Condicion\nHora                   \n9     Patchy light rain\n\n¡Que tengas un excelente día! ☔️😊\n'

In [9]:
PHONE_NUMBER

'+1 659 251 8445'

# Mensaje SMS desde Twilio

In [15]:
df_rain

,Condicion
Hora,
9,Patchy light rain


In [16]:
import time
from twilio.rest import Client

time.sleep(2)  # Wait for 2 seconds

# Get Twilio credentials
account_sid = TWILIO_ACCOUNT_SID
auth_token = TWILIO_AUTH_TOKEN

# Initialize Twilio client
client = Client(account_sid, auth_token)

# Compose the rain forecast message
try:
    if not df_rain.empty:
        # df_rain tiene 'Hora' como índice, así que lo usamos adecuadamente
        horas_lluvia = [
            f"Hora: {hora}, Condición: {row['Condicion']}"
            for hora, row in df_rain.iterrows()
        ]
        horas_lluvia_str = "\n".join(horas_lluvia)
    else:
        horas_lluvia_str = 'No se esperan lluvias importantes durante el día.'
except Exception as e:
    horas_lluvia_str = f"(Error al generar listado de lluvias: {e})"

sms_body = (
    f"Pronostico de lluvias para hoy, {df['Fecha'][0]}, en {query}:\n"
    f"Horas con lluvia pronosticada:\n"
    f"{horas_lluvia_str}"
)

# Send the message via Twilio
try:
    message = client.messages.create(
        body=sms_body,
        from_=PHONE_NUMBER,
        to='+51923046453'  # Update this number as needed
    )
    print('Mensaje Enviado ' + message.sid)
except Exception as e:
    print(f"Error al enviar el mensaje: {e}")

Mensaje Enviado SMa41b9f6127f4e6c2c10940435d51206b


In [13]:
sms_body

'Pronostico de lluvias para hoy, 2026-07-07, en Tarapoto: Horas con lluvia pronosticada: Patchy light rain'

# Challenge 

* Extrae el valor del dolar en tu país y el top 10 de criptomonedas con su respectiva valoración
* Ahora envia un mensaje diarío a tu Whatsapp usando Twilio

**hint 💡** Investiga que API's gratuitas existen para consultar estos datos



<img src="WhatsApp Image 2022-09-13 at 9.12.18 AM.jpeg" width="200" height="200" />

In [1]:
import requests

def obtener_tasa_cambio(apikey: str) -> dict:
    """
    Obtiene las tasas de cambio actualizadas desde CurrencyFreaks API.

    Args:
        apikey (str): Tu API key para CurrencyFreaks.

    Returns:
        dict: Un diccionario con la respuesta del API (o None si falla).
    """
    url = f"https://api.currencyfreaks.com/v2.0/rates/latest?apikey={apikey}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error al obtener tasas de cambio: {e}")
        return None

In [2]:
obtener_tasa_cambio('25e5772e21d14bc99c291e2386f2c45a')

{'date': '2026-07-09 00:00:00+00',
 'base': 'USD',
 'rates': {'AGLD': '6.648936170212766',
  'FJD': '2.25175',
  'SCR': '14.579',
  'BBD': '2.0',
  'HNL': '26.7554',
  'UGX': '3673.02',
  'COSMOSDYDX': '7.15307582260372',
  'NEAR': '0.5291005291005291',
  'AIOZ': '19.455252918287936',
  'AUDIO': '73.86138990479962',
  'WLD': '2.576655501159495',
  'HNT': '4.504504504504505',
  'ETHFI': '2.5906735751295336',
  'FARM': '0.1923076923076923',
  'A8': '185.1851851851852',
  'SDG': '600.171',
  'DGB': '411.1270498813674',
  'AB': '1008.2049434983818',
  'BCH': '0.0042526047203912',
  'AI': '37.67897513187641',
  'FKP': '0.746631',
  'JST': '10.547529399865704',
  'HOT': '3073.790400046875',
  'AO': '0.5002075884370307',
  'AR': '0.5133886680966515',
  'B3': '2100.840336134454',
  'CHILLGUY': '94.82086144586037',
  'SEI': '21.222410865874362',
  'SEK': '9.69495',
  'LION': '669.7403447387754',
  'BB': '54.06314363611999',
  'QAR': '3.64525',
  'JTO': '1.571338780641106',
  'WEMIX': '3.5585912